# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Dataset published: {metadata['datePublished']}, Dataset version: {metadata['version']}")
print("Keywords:", metadata['keywords'])

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will list all record sets, and for each, print available fields and columns by their `@id`.

In [ ]:
# Identify record sets from metadata
# mlcroissant exposes record_sets via dataset.record_sets
record_sets = [r for r in dataset.record_sets]
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']} | Name: {rs.get('name','(no name)')} | Description: {rs.get('description','')}")
    # list fields
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        print(f"   - Field @id: {f['@id']} | Name: {f.get('name','(no name)')} | Data Type: {f.get('dataType','')} | Description: {f.get('description','')}")
    # list columns
    columns = rs.get('column', [])
    if columns:
        if not isinstance(columns, list):
            columns = [columns]
        for c in columns:
            print(f"   - Column @id: {c['@id']} | Name: {c.get('name','(no name)')} | Data Type: {c.get('dataType','')}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

You can select specific record sets using their `@id`. For this dataset, we will extract from all record sets demonstrated above.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Prepare list of record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set {rs_id} with {len(df)} rows and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Pick first record set for demonstration
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print("\nPreview of main DataFrame:")
    print(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by specific criteria, normalizing numeric fields, and grouping data.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with candidate field or column `@id`s from step 2. For demonstration, we attempt to find a numeric survey score (such as `gad_7_score`, `phq_9_score`, or `psq_score`) and group by demographic, for example, `gender`.

In [ ]:
# Attempt to find a main numeric field
df = dataframes[main_rs_id]

# Try candidate field/column names
candidate_numeric_ids = [col for col in df.columns if 'score' in col.lower() or 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower()]
print("Numeric fields found:", candidate_numeric_ids)
numeric_field = candidate_numeric_ids[0] if candidate_numeric_ids else df.columns[0]

print(f"Using numeric field: {numeric_field}")

# Filter for values above threshold
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by demographic
    candidate_group_ids = [col for col in df.columns if 'gender' in col.lower() or 'age' in col.lower() or 'village' in col.lower()]
    group_field = candidate_group_ids[0] if candidate_group_ids else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_score')
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset: we plot the distribution of the numeric field and compare mean score by demographic group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        # Only use group values with enough entries
        group_counts = df[group_field].value_counts()
        top_groups = group_counts.index[:5]
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
We loaded and explored the FAIR² mental health survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

- We identified available record sets, fields, and their IDs via the Croissant schema.
- Data was loaded directly from record sets referenced by their `@id`.
- We extracted and analyzed core numeric mental health scores, filtered and normalized values, and visualized demographic differences.
- This notebook can be extended for further AI modeling, statistical analysis, or more advanced data processing using the standardized Croissant schema.
